# 노트북 2. System Prompt vs User Prompt + LangSmith

> Phase 1 — 기초: LLM과 대화하는 법

LLM은 '역할(role)'에 따라 메시지를 다르게 취급합니다.
이 구조를 이해해야 챗봇의 행동을 제어할 수 있습니다.
그리고 그 제어가 실제로 잘 되는지 추적하는 도구가 LangSmith입니다.

**학습 목표**
- 메시지 역할 체계(system / user / assistant)를 이해한다
- google-genai의 system_instruction과 LangChain의 SystemMessage를 사용할 수 있다
- 효과적인 System Prompt를 설계하는 원칙을 적용할 수 있다
- LangSmith를 연동하여 LLM 호출을 추적하고 분석할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | 메시지 역할 + System Prompt 설계 + LangSmith 연동 | 읽고 실행 |
| Part 2 — 실습 | System Prompt 작성 + 비교 실험 + LangSmith 트레이싱 | 직접 작성 |
| Part 3 — 챌린지 | 도메인 전문가 챗봇 설계 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai langsmith

import os
from google import genai

# API 키 설정 (Colab 환경 기준)
# 왼쪽 키 아이콘 → GEMINI_API_KEY 등록 후 실행
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

# LangChain 모델도 미리 생성
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_API_KEY,
)

print("환경 설정 완료")

---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 메시지 역할 체계

LLM API는 단순히 "질문 → 답변"이 아니라, **역할(role)이 부여된 메시지들의 리스트**를 입력으로 받습니다.
각 메시지에는 "이 메시지가 누구의 것인지"를 나타내는 역할이 붙어 있습니다.

| 역할 | 설명 | 비유 |
|------|------|------|
| **system** | 모델의 행동 규칙을 정의. 사용자에게는 보이지 않는 "무대 뒤 지시" | 연극의 연출 노트 |
| **user** (human) | 실제 사용자가 보내는 입력 | 관객의 질문 |
| **assistant** (model/AI) | 모델이 이전에 생성한 응답 | 배우의 대사 |

> 핵심: system 메시지는 모델의 "성격"과 "능력 범위"를 결정합니다.
> 같은 모델이라도 system 메시지를 바꾸면 완전히 다른 챗봇이 됩니다.

### google-genai에서의 역할 표현

google-genai SDK에서는 역할을 다음과 같이 표현합니다:

- `system`: `system_instruction` 파라미터로 별도 전달
- `user`: `contents`에서 `role="user"`
- `model`: `contents`에서 `role="model"` (assistant에 해당)

아래 코드는 google-genai의 메시지 구조를 보여줍니다.

In [ ]:
from google.genai.types import Content, Part

# google-genai에서 역할이 부여된 메시지 구조
# system은 별도 파라미터, user/model은 contents 리스트에 배치
response = client.models.generate_content(
    model="gemini-2.5-flash",
    config={"system_instruction": "당신은 친절한 한국어 도우미입니다."},
    contents=[
        Content(role="user", parts=[Part(text="안녕하세요?")]),
    ],
)

print(response.text)

단순한 호출에서는 `contents`에 문자열만 전달해도 됩니다.
그러면 자동으로 `user` 역할로 처리됩니다.

In [ ]:
# 단순 문자열 전달 — 자동으로 user 역할
response_simple = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="안녕하세요?",
)

print(response_simple.text)

### LangChain에서의 역할 표현

LangChain은 역할별로 전용 메시지 클래스를 제공합니다:

| 역할 | LangChain 클래스 | google-genai 대응 |
|------|-----------------|------------------|
| system | `SystemMessage` | `system_instruction` |
| user | `HumanMessage` | `role="user"` |
| assistant | `AIMessage` | `role="model"` |

아래 코드는 LangChain 메시지 객체를 사용한 호출을 보여줍니다.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# LangChain에서 역할별 메시지 구성
messages = [
    SystemMessage(content="당신은 친절한 한국어 도우미입니다."),
    HumanMessage(content="안녕하세요?"),
]

response_lc = model.invoke(messages)
print(f"응답 타입: {type(response_lc).__name__}")  # AIMessage
print(f"응답: {response_lc.content}")

이전 대화 맥락을 전달하려면 `AIMessage`도 리스트에 포함합니다.
(멀티턴 대화의 상세한 원리는 노트북 3에서 다룹니다.)

아래 코드는 3개 역할이 모두 포함된 메시지 리스트의 예시를 보여줍니다.

In [ ]:
# 3개 역할이 모두 포함된 대화 이력
messages_with_history = [
    SystemMessage(content="당신은 수학 교사입니다. 풀이 과정을 보여주세요."),
    HumanMessage(content="3 + 5는?"),
    AIMessage(content="3 + 5 = 8입니다."),         # 이전 모델 응답
    HumanMessage(content="거기에 2를 곱하면?"),     # 현재 질문
]

response_history = model.invoke(messages_with_history)
print(response_history.content)

> 핵심: 모델은 메시지 리스트의 역할을 보고 "누가 말한 것인지"를 구분합니다.
> system은 모델의 행동 규칙, user는 사용자 입력, assistant는 모델의 이전 응답입니다.
> 이 구조를 이해하면 챗봇의 행동을 정밀하게 제어할 수 있습니다.

## 1.2 System Prompt: google-genai vs LangChain

**System Prompt**(시스템 프롬프트)는 모델에게 "너는 이런 존재이고, 이렇게 행동해라"를 지시하는 메시지입니다.
사용자에게는 보이지 않지만 모델의 모든 응답에 영향을 미칩니다.

두 방식에서 system prompt를 설정하는 방법이 다릅니다.

### google-genai: system_instruction

google-genai에서는 `config` 딕셔너리의 `system_instruction` 키로 전달합니다.

아래 코드는 system_instruction을 사용하여 모델에게 역할을 부여하는 예시를 보여줍니다.

In [ ]:
# google-genai: system_instruction으로 역할 부여
response_poet = client.models.generate_content(
    model="gemini-2.5-flash",
    config={
        "system_instruction": "당신은 시인입니다. 모든 답변을 시(poem) 형태로 작성하세요.",
    },
    contents="봄에 대해 알려주세요.",
)

print(response_poet.text)

system_instruction 없이 동일한 질문을 하면 어떻게 다른지 비교해봅시다.

In [ ]:
# system_instruction 없이 동일 질문
response_no_system = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="봄에 대해 알려주세요.",
)

print(response_no_system.text)

### LangChain: SystemMessage

LangChain에서는 `SystemMessage`를 메시지 리스트의 첫 번째에 배치합니다.
또는 `ChatPromptTemplate.from_messages()`에서 `("system", "내용")` 튜플로 정의합니다.

아래 코드는 두 가지 방법을 모두 보여줍니다.

In [ ]:
# 방법 1: SystemMessage 객체 직접 사용
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="당신은 시인입니다. 모든 답변을 시(poem) 형태로 작성하세요."),
    HumanMessage(content="봄에 대해 알려주세요."),
]

response_poet_lc = model.invoke(messages)
print("[방법 1: SystemMessage 객체]")
print(response_poet_lc.content)

In [ ]:
# 방법 2: ChatPromptTemplate.from_messages()에서 튜플로 정의
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 시인입니다. 모든 답변을 시(poem) 형태로 작성하세요."),
    ("human", "{question}"),
])

chain = prompt | model | StrOutputParser()
result = chain.invoke({"question": "봄에 대해 알려주세요."})

print("[방법 2: ChatPromptTemplate 튜플]")
print(result)

> 핵심: google-genai는 `system_instruction` 파라미터로 분리하고,
> LangChain은 `SystemMessage`를 메시지 리스트에 포함합니다.
> 동작은 동일합니다. 모델에게 "이렇게 행동해라"를 지시하는 것입니다.

### System Prompt의 효과 확인: 동일 질문, 다른 성격

system prompt만 바꿔도 응답의 톤, 길이, 포맷이 완전히 달라진다는 것을 직접 확인해봅시다.

아래 코드는 동일한 질문에 3가지 다른 system prompt를 적용하여 결과를 비교합니다.

In [ ]:
# 동일 질문, 3가지 다른 system prompt
question = "인공지능이 일자리에 미치는 영향은?"

system_prompts = {
    "낙관론자": "당신은 기술 낙관론자입니다. 기술 발전의 긍정적 측면을 강조하세요. 2문장 이내로 답변하세요.",
    "비관론자": "당신은 기술 비관론자입니다. 기술 발전의 위험성을 경고하세요. 2문장 이내로 답변하세요.",
    "중립 분석가": "당신은 중립적인 분석가입니다. 양쪽 관점을 균형 있게 제시하세요. 2문장 이내로 답변하세요.",
}

for persona, system_prompt in system_prompts.items():
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        config={"system_instruction": system_prompt},
        contents=question,
    )
    print(f"[{persona}]")
    print(response.text)
    print()

같은 모델, 같은 질문인데 system prompt에 따라 관점이 완전히 달라진 것을 확인할 수 있습니다.
이것이 system prompt의 핵심 가치입니다.

## 1.3 System Prompt 설계 원칙

좋은 System Prompt는 세 가지 요소로 구성됩니다:

1. **페르소나(Persona)** — 모델이 어떤 존재인지 정의
2. **제약 조건(Constraints)** — 반드시 지켜야 할 규칙
3. **출력 포맷(Format)** — 응답의 형태를 지정

### 페르소나 정의

"너는 ~이다"로 시작하는 역할 정의입니다.
모델은 이 정의에 맞춰 어휘, 톤, 지식 범위를 조절합니다.

아래 코드는 페르소나에 따른 응답 차이를 보여줍니다.

In [ ]:
# 페르소나에 따른 응답 차이
question = "물이 끓는 이유를 설명해주세요."

# 초등학교 교사
resp_teacher = client.models.generate_content(
    model="gemini-2.5-flash",
    config={"system_instruction": "당신은 초등학교 3학년 담임 선생님입니다. 아이들이 이해할 수 있도록 쉽게 설명하세요."},
    contents=question,
)
print("[초등학교 교사]")
print(resp_teacher.text)
print()

# 물리학 교수
resp_professor = client.models.generate_content(
    model="gemini-2.5-flash",
    config={"system_instruction": "당신은 서울대학교 물리학과 교수입니다. 학술적으로 정확하게 설명하세요."},
    contents=question,
)
print("[물리학 교수]")
print(resp_professor.text)

### 제약 조건

"반드시 ~해야 한다", "절대 ~하지 마라" 형태의 규칙입니다.
모델의 행동 범위를 좁혀서 예측 가능한 응답을 만듭니다.

아래 코드는 제약 조건의 효과를 보여줍니다.

In [ ]:
# 제약 조건 예시
system_with_constraints = """
당신은 고객 상담 챗봇입니다.

규칙:
- 반드시 존댓말을 사용하세요.
- 모든 답변은 3문장 이내로 작성하세요.
- 가격이나 할인에 대한 질문에는 "담당자에게 연결해드리겠습니다"로 답변하세요.
- 경쟁사 제품에 대한 비교 질문에는 답변을 거절하세요.
""".strip()

# 일반 질문
resp1 = client.models.generate_content(
    model="gemini-2.5-flash",
    config={"system_instruction": system_with_constraints},
    contents="배송은 보통 얼마나 걸리나요?",
)
print("[일반 질문]")
print(resp1.text)
print()

# 가격 질문 — 제약 조건에 의해 담당자 연결 안내
resp2 = client.models.generate_content(
    model="gemini-2.5-flash",
    config={"system_instruction": system_with_constraints},
    contents="이 제품 할인 중인가요?",
)
print("[가격 질문]")
print(resp2.text)

### 출력 포맷 지정

"JSON으로 응답", "3문장 이내", "번호를 매겨서" 등 출력 형태를 명시합니다.
후속 코드에서 응답을 파싱해야 할 때 특히 중요합니다.

아래 코드는 포맷 지정에 따른 응답 차이를 보여줍니다.

In [ ]:
# 포맷 지정 없이
resp_no_format = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="파이썬의 장점 3가지",
)
print("[포맷 지정 없음]")
print(resp_no_format.text)
print()

In [ ]:
# 포맷을 명시적으로 지정
resp_with_format = client.models.generate_content(
    model="gemini-2.5-flash",
    config={
        "system_instruction": (
            "다음 규칙에 따라 답변하세요:\n"
            "1. 번호를 매겨서 나열할 것\n"
            "2. 각 항목은 한 줄로 작성할 것\n"
            "3. 부연 설명 없이 핵심만 작성할 것"
        ),
    },
    contents="파이썬의 장점 3가지",
)
print("[포맷 지정]")
print(resp_with_format.text)

### 좋은 System Prompt vs 나쁜 System Prompt

좋은 system prompt는 구체적이고 명확합니다. 나쁜 system prompt는 모호하거나 모순됩니다.

아래 코드는 같은 의도를 가진 system prompt를 두 가지 품질로 비교합니다.

In [ ]:
question = "회사에서 팀원과 갈등이 생겼어요. 어떻게 하면 좋을까요?"

# 나쁜 예: 모호하고 구체성 없음
bad_prompt = "좋은 상담사가 되어줘."

resp_bad = client.models.generate_content(
    model="gemini-2.5-flash",
    config={"system_instruction": bad_prompt},
    contents=question,
)
print("[나쁜 System Prompt]")
print(f"프롬프트: {bad_prompt}")
print(f"응답 길이: {len(resp_bad.text)}자")
print(resp_bad.text[:300])
print("...\n")

In [ ]:
# 좋은 예: 페르소나 + 제약 + 포맷이 명확
good_prompt = """
당신은 10년 경력의 조직심리학 전문 상담사입니다.

규칙:
- 먼저 상대방의 감정을 인정하는 문장으로 시작하세요.
- 구체적인 행동 단계를 3가지 제안하세요.
- 각 단계를 번호로 매기고 한 줄로 작성하세요.
- 마지막에 격려 문장을 추가하세요.
- 전체 답변은 200자 이내로 작성하세요.
""".strip()

resp_good = client.models.generate_content(
    model="gemini-2.5-flash",
    config={"system_instruction": good_prompt},
    contents=question,
)
print("[좋은 System Prompt]")
print(f"응답 길이: {len(resp_good.text)}자")
print(resp_good.text)

> 핵심: 좋은 System Prompt의 3요소
>
> 1. **페르소나**: "~입니다" (역할과 전문성 정의)
> 2. **제약 조건**: "반드시/절대" (행동 범위 제한)
> 3. **출력 포맷**: "~형태로" (응답 구조 지정)
>
> 이 세 가지가 구체적일수록 모델의 응답이 예측 가능하고 일관됩니다.

### LangChain에서 System Prompt 활용 패턴

LangChain의 `ChatPromptTemplate.from_messages()`를 사용하면
system prompt에도 변수를 넣을 수 있습니다.
이를 통해 하나의 템플릿으로 다양한 페르소나를 구현할 수 있습니다.

아래 코드는 system prompt에 변수를 사용하는 패턴을 보여줍니다.

In [ ]:
# system prompt에 변수를 넣는 패턴
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {specialty} 전문가입니다. {style}으로 답변하세요."),
    ("human", "{question}"),
])

chain = prompt_template | model | StrOutputParser()

# 요리 전문가
result1 = chain.invoke({
    "specialty": "한식 요리",
    "style": "레시피 단계별 형태",
    "question": "김치찌개 만드는 법을 알려주세요.",
})
print("[한식 요리 전문가]")
print(result1)
print()

In [ ]:
# 같은 체인, 다른 변수 — 법률 전문가
result2 = chain.invoke({
    "specialty": "노동법",
    "style": "법률 용어를 쉽게 풀어서",
    "question": "연차 사용을 회사가 거부할 수 있나요?",
})
print("[노동법 전문가]")
print(result2)

## 1.4 LangSmith 연동

**LangSmith**는 LangChain 팀이 만든 LLM 개발 플랫폼입니다.
LLM 호출을 자동으로 **트레이싱(tracing)**하여, 실제로 어떤 프롬프트가 전송되었고
어떤 응답이 돌아왔는지를 대시보드에서 확인할 수 있습니다.

트레이싱이 중요한 이유:
- system prompt를 바꿨을 때 "정말 의도대로 동작하는지"를 **데이터**로 확인
- "느낌"이 아니라 "근거"로 프롬프트를 개선 가능
- 토큰 사용량, 응답 시간, 비용 추정을 자동으로 기록

> LangSmith는 https://smith.langchain.com 에서 무료로 가입할 수 있습니다.
> 가입 후 API 키를 발급받으세요.

### LangSmith 환경변수 설정

LangSmith를 활성화하려면 환경변수 3개를 설정하면 됩니다.
설정만 하면 모든 LangChain 호출이 자동으로 트레이싱됩니다.

아래 코드는 LangSmith 환경변수를 설정합니다.
Colab 보안 키에 `LANGSMITH_API_KEY`를 미리 등록해두세요.

In [ ]:
# LangSmith 환경변수 설정
# LANGSMITH_API_KEY를 Colab 보안 키에 등록한 후 실행하세요
# (LangSmith 계정이 없다면 이 셀을 건너뛰어도 됩니다)

try:
    LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")
    os.environ["LANGCHAIN_TRACING_V2"] = "true"           # 트레이싱 활성화
    os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY    # API 키
    os.environ["LANGCHAIN_PROJECT"] = "note-02-prompt"     # 프로젝트명
    print("LangSmith 연동 완료")
    print(f"프로젝트: {os.environ['LANGCHAIN_PROJECT']}")
    LANGSMITH_ENABLED = True
except Exception:
    print("LANGSMITH_API_KEY가 등록되지 않았습니다.")
    print("LangSmith 관련 셀은 건너뛰어도 학습에 지장 없습니다.")
    LANGSMITH_ENABLED = False

### 트레이싱 확인

환경변수를 설정한 후 LangChain으로 호출하면, 해당 호출이 LangSmith 대시보드에 기록됩니다.

아래 코드를 실행한 후, https://smith.langchain.com 에서 `note-02-prompt` 프로젝트를 열어보세요.
호출 내역이 기록되어 있을 것입니다.

In [ ]:
# LangSmith에 트레이싱될 호출
# 이 셀 실행 후 LangSmith 대시보드에서 확인하세요
traced_response = model.invoke([
    SystemMessage(content="당신은 파이썬 전문가입니다. 간결하게 답변하세요."),
    HumanMessage(content="리스트 컴프리헨션이란 무엇인가요?"),
])

print(traced_response.content)

### LangSmith 대시보드에서 볼 수 있는 것들

LangSmith 대시보드에서는 각 호출에 대해 다음 정보를 확인할 수 있습니다:

| 항목 | 설명 |
|------|------|
| Input | 실제 전송된 프롬프트 전문 (system + user 메시지) |
| Output | 모델의 응답 전문 |
| Tokens | 입력/출력 토큰 수 |
| Latency | 응답 시간 (ms) |
| Cost | 비용 추정 |
| Model | 사용된 모델명과 버전 |

이 정보를 활용하면:
- system prompt를 변경한 전후 비교가 가능합니다
- 어떤 질문에 토큰이 많이 소모되는지 파악할 수 있습니다
- 응답 시간이 느린 호출을 식별할 수 있습니다

### 체인 호출도 트레이싱됨

LCEL 체인을 실행하면 체인 내부의 각 단계(prompt → model → parser)가
개별적으로 트레이싱됩니다. 체인의 어느 단계에서 시간이 걸리는지 파악할 수 있습니다.

아래 코드는 체인 호출을 트레이싱하는 예시를 보여줍니다.

In [ ]:
# 체인 호출 — LangSmith에서 각 단계별 트레이싱 확인 가능
tracing_chain = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. 한 문장으로 답변하세요."),
    ("human", "{question}"),
]) | model | StrOutputParser()

result = tracing_chain.invoke({
    "role": "역사학자",
    "question": "한글은 누가 만들었나요?",
})
print(result)
print("\n(LangSmith 대시보드에서 이 호출의 상세 트레이스를 확인해보세요)")

### 프로젝트 분리로 실험 관리

LangSmith에서는 `LANGCHAIN_PROJECT` 환경변수로 프로젝트를 분리할 수 있습니다.
system prompt A와 B를 비교하는 실험이라면, 각각 다른 프로젝트에 기록하면 비교가 쉬워집니다.

아래 코드는 프로젝트를 바꿔가며 호출하는 예시를 보여줍니다.

In [ ]:
# 프로젝트 A — 짧은 답변 스타일
os.environ["LANGCHAIN_PROJECT"] = "note-02-short-style"

short_chain = ChatPromptTemplate.from_messages([
    ("system", "한 문장으로 답변하세요."),
    ("human", "{question}"),
]) | model | StrOutputParser()

result_short = short_chain.invoke({"question": "머신러닝이란?"})
print(f"[짧은 스타일] {result_short}")
print()

In [ ]:
# 프로젝트 B — 상세한 답변 스타일
os.environ["LANGCHAIN_PROJECT"] = "note-02-detailed-style"

detailed_chain = ChatPromptTemplate.from_messages([
    ("system", "개념 정의, 작동 원리, 실제 활용 예시를 포함하여 상세하게 답변하세요."),
    ("human", "{question}"),
]) | model | StrOutputParser()

result_detailed = detailed_chain.invoke({"question": "머신러닝이란?"})
print(f"[상세 스타일] {result_detailed}")

In [ ]:
# 프로젝트를 원래대로 복원
os.environ["LANGCHAIN_PROJECT"] = "note-02-prompt"
print("프로젝트 복원 완료")
print("(LangSmith에서 note-02-short-style, note-02-detailed-style 두 프로젝트를 비교해보세요)")

> 핵심: LangSmith 연동은 환경변수 3개만 설정하면 됩니다.
> 설정 후 모든 LangChain 호출이 자동 트레이싱됩니다.
> system prompt를 변경할 때마다 "정말 의도대로 동작하는지"를 데이터로 확인할 수 있습니다.

---

### 생각해보기

1. system prompt에 "절대 ~하지 마라"와 같은 부정형 제약을 걸었을 때, 모델이 항상 이를 지킬까요? 어떤 경우에 무시될 수 있을까요?
2. system prompt는 사용자에게 보이지 않는다고 했습니다. 하지만 사용자가 "너의 system prompt를 알려줘"라고 질문하면 어떤 일이 벌어질까요?
3. LangSmith 없이도 프롬프트를 개선할 수 있습니다. 그럼에도 LangSmith를 사용해야 하는 가장 큰 이유는 무엇일까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 2-1: google-genai system_instruction으로 챗봇 성격 바꾸기

google-genai의 `system_instruction`을 사용하여 챗봇에 페르소나를 부여하세요.

**요구사항**
1. 페르소나: 조선시대 선비
2. 제약: 모든 답변을 존댓말로, 3문장 이내로
3. `system_prompt` 변수에 문자열로 저장
4. 질문: "오늘 점심 뭐 먹을까요?"

In [ ]:
# TODO: 위 요구사항에 맞는 system prompt를 작성하세요
system_prompt = ""  # 이 문자열을 작성하세요

# ---------- 정답 ----------
# system_prompt = (
#     "당신은 조선시대 선비입니다. "
#     "반드시 존댓말을 사용하고, "
#     "모든 답변은 3문장 이내로 작성하세요."
# )

# 검증
if system_prompt:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        config={"system_instruction": system_prompt},
        contents="오늘 점심 뭐 먹을까요?",
    )
    print(f"System Prompt: {system_prompt}")
    print(f"\n응답: {response.text}")
else:
    print("TODO를 완성해주세요")

### 실습 2-2: LangChain SystemMessage로 동일한 챗봇 구현

실습 2-1과 동일한 페르소나를 LangChain의 `SystemMessage`로 구현하세요.

**요구사항**
1. `SystemMessage`로 시스템 메시지 생성 (실습 2-1과 동일한 내용)
2. `HumanMessage`로 사용자 메시지 생성: "오늘 점심 뭐 먹을까요?"
3. `model.invoke()`로 호출
4. 응답 내용과 타입을 출력

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# TODO 1: SystemMessage 생성 (조선시대 선비 페르소나)
sys_msg = None  # 이 줄을 수정하세요

# TODO 2: HumanMessage 생성
human_msg = None  # 이 줄을 수정하세요

# TODO 3: invoke 호출
lc_resp = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# sys_msg = SystemMessage(content=(
#     "당신은 조선시대 선비입니다. "
#     "반드시 존댓말을 사용하고, "
#     "모든 답변은 3문장 이내로 작성하세요."
# ))
# human_msg = HumanMessage(content="오늘 점심 뭐 먹을까요?")
# lc_resp = model.invoke([sys_msg, human_msg])

# 검증
if lc_resp is not None:
    print(f"응답 타입: {type(lc_resp).__name__}")
    print(f"응답: {lc_resp.content}")
else:
    print("TODO를 완성해주세요")

### 실습 2-3: 동일 질문에 다른 System Prompt 비교

동일한 질문에 3가지 다른 system prompt를 적용하고 결과를 비교하세요.

**요구사항**
1. 질문: "커피가 건강에 미치는 영향은?"
2. 3가지 system prompt를 딕셔너리로 정의:
   - 키: 페르소나 이름, 값: system prompt 문자열
   - 페르소나 1: 의사 (건강 관점, 2문장 이내)
   - 페르소나 2: 바리스타 (커피 애호가 관점, 2문장 이내)
   - 페르소나 3: 영양사 (영양학 관점, 2문장 이내)
3. 반복문으로 3가지 결과를 모두 출력

In [ ]:
question = "커피가 건강에 미치는 영향은?"

# TODO: 3가지 system prompt를 딕셔너리로 정의하세요
personas = {}  # 이 딕셔너리를 채우세요

# ---------- 정답 ----------
# personas = {
#     "의사": "당신은 내과 전문의입니다. 의학적 근거를 바탕으로 건강 관점에서 답변하세요. 2문장 이내로 작성하세요.",
#     "바리스타": "당신은 10년 경력의 바리스타입니다. 커피 애호가의 관점에서 긍정적으로 답변하세요. 2문장 이내로 작성하세요.",
#     "영양사": "당신은 임상 영양사입니다. 영양학적 관점에서 균형 잡힌 의견을 제시하세요. 2문장 이내로 작성하세요.",
# }

# 검증
if personas:
    for name, sys_prompt in personas.items():
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            config={"system_instruction": sys_prompt},
            contents=question,
        )
        print(f"[{name}] {resp.text}")
        print()
else:
    print("TODO를 완성해주세요")

### 실습 2-4: System Prompt로 출력 포맷 제어

System Prompt를 사용하여 모델의 출력을 특정 포맷으로 강제하세요.

**요구사항**
1. `ChatPromptTemplate.from_messages()`를 사용하여 LCEL 체인 구성
2. system 메시지에 다음 포맷 규칙을 포함:
   - 항목은 "- " (하이픈)으로 시작할 것
   - 각 항목은 "장점:" 또는 "단점:" 으로 시작할 것
   - 장점 3개, 단점 2개를 반드시 포함할 것
3. 질문 변수: `{topic}`
4. `"원격 근무"`를 입력으로 체인 실행

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO 1: system 메시지에 포맷 규칙을 포함하는 프롬프트 템플릿 생성
format_prompt = None  # 이 줄을 수정하세요

# TODO 2: 체인 구성
format_chain = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# format_prompt = ChatPromptTemplate.from_messages([
#     ("system", (
#         "다음 포맷 규칙에 따라 답변하세요:\n"
#         "- 각 항목은 '- '(하이픈)으로 시작할 것\n"
#         "- 각 항목은 '장점:' 또는 '단점:'으로 시작할 것\n"
#         "- 장점 3개, 단점 2개를 반드시 포함할 것\n"
#         "- 각 항목은 한 줄로 작성할 것"
#     )),
#     ("human", "{topic}의 장단점을 알려주세요."),
# ])
# format_chain = format_prompt | model | StrOutputParser()

# 검증
if format_chain is not None:
    result = format_chain.invoke({"topic": "원격 근무"})
    print(result)
else:
    print("TODO를 완성해주세요")

### 실습 2-5: 페르소나 변수화 체인 구성

system prompt에 변수를 사용하여, 하나의 체인으로 다양한 페르소나를 구현하세요.

**요구사항**
1. `ChatPromptTemplate.from_messages()`를 사용
2. system 메시지: "당신은 {persona}입니다. {constraint}"
3. human 메시지: "{question}"
4. 체인 구성 후 두 가지 입력으로 각각 실행:
   - 입력 1: `persona="해적 선장"`, `constraint="해적 말투로 답변하세요."`, `question="오늘 날씨 어때요?"`
   - 입력 2: `persona="뉴스 앵커"`, `constraint="객관적이고 격식 있게 답변하세요."`, `question="오늘 날씨 어때요?"`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO 1: 변수화된 system prompt를 포함하는 프롬프트 템플릿
persona_prompt = None  # 이 줄을 수정하세요

# TODO 2: 체인 구성
persona_chain = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# persona_prompt = ChatPromptTemplate.from_messages([
#     ("system", "당신은 {persona}입니다. {constraint}"),
#     ("human", "{question}"),
# ])
# persona_chain = persona_prompt | model | StrOutputParser()

# 검증
if persona_chain is not None:
    # 해적 선장
    r1 = persona_chain.invoke({
        "persona": "해적 선장",
        "constraint": "해적 말투로 답변하세요.",
        "question": "오늘 날씨 어때요?",
    })
    print(f"[해적 선장] {r1}\n")

    # 뉴스 앵커
    r2 = persona_chain.invoke({
        "persona": "뉴스 앵커",
        "constraint": "객관적이고 격식 있게 답변하세요.",
        "question": "오늘 날씨 어때요?",
    })
    print(f"[뉴스 앵커] {r2}")
else:
    print("TODO를 완성해주세요")

### 실습 2-6: LangSmith 트레이스 확인

LangSmith 환경변수를 설정하고, 호출이 트레이싱되는지 확인하세요.
(LangSmith 계정이 없다면 환경변수 설정 부분만 이해하고 넘어가세요.)

**요구사항**
1. `LANGCHAIN_PROJECT`를 `"note-02-practice"`로 설정
2. LCEL 체인을 구성하여 호출
3. 호출 후 LangSmith 대시보드에서 트레이스 확인
4. 프로젝트명을 원래대로 복원

In [ ]:
# TODO 1: LANGCHAIN_PROJECT 환경변수를 "note-02-practice"로 설정
# (한 줄이면 됩니다)

# ---------- 정답 ----------
# os.environ["LANGCHAIN_PROJECT"] = "note-02-practice"

# TODO 2: 아무 체인이나 구성하여 호출
practice_chain = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# practice_chain = (
#     ChatPromptTemplate.from_messages([
#         ("system", "한 문장으로 답변하세요."),
#         ("human", "{question}"),
#     ])
#     | model
#     | StrOutputParser()
# )

# 검증
if practice_chain is not None:
    result = practice_chain.invoke({"question": "LangSmith란 무엇인가요?"})
    print(result)
    print("\n(LangSmith 대시보드에서 'note-02-practice' 프로젝트를 확인해보세요)")
else:
    print("TODO를 완성해주세요")

In [ ]:
# 프로젝트명 복원
os.environ["LANGCHAIN_PROJECT"] = "note-02-prompt"
print("프로젝트 복원 완료")

### 실습 2-7: 복합 System Prompt 작성

페르소나 + 제약 + 포맷을 모두 포함하는 종합적인 System Prompt를 작성하세요.

**요구사항**
1. 페르소나: IT 회사의 기술 면접관
2. 제약:
   - 답변자의 수준을 고려하여 후속 질문을 하나 더 제시할 것
   - 정답을 직접 알려주지 말 것
   - 존댓말 사용
3. 포맷:
   - 먼저 답변에 대한 피드백 (1-2문장)
   - 그 다음 후속 질문 ("후속 질문:" 으로 시작)
4. google-genai의 `system_instruction`으로 적용
5. 질문: "해시 테이블의 시간 복잡도를 알려주세요"

In [ ]:
# TODO: 페르소나 + 제약 + 포맷을 포함하는 system prompt 작성
interviewer_prompt = ""  # 이 문자열을 작성하세요

# ---------- 정답 ----------
# interviewer_prompt = (
#     "당신은 IT 회사의 기술 면접관입니다.\n"
#     "\n"
#     "규칙:\n"
#     "- 답변자의 수준을 고려하여 후속 질문을 하나 더 제시하세요.\n"
#     "- 정답을 직접 알려주지 마세요. 힌트만 제공하세요.\n"
#     "- 존댓말을 사용하세요.\n"
#     "\n"
#     "답변 포맷:\n"
#     "1. 답변에 대한 피드백 (1-2문장)\n"
#     "2. '후속 질문:' 으로 시작하는 후속 질문"
# )

# 검증
if interviewer_prompt:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        config={"system_instruction": interviewer_prompt},
        contents="해시 테이블의 시간 복잡도를 알려주세요",
    )
    print(resp.text)
else:
    print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 2-3에서 같은 질문에 페르소나만 바꿨는데 답변이 달라졌습니다. 모델은 실제로 '의사'나 '바리스타'가 된 것일까요, 아니면 흉내를 내는 것일까요? 그 차이가 중요한가요?
2. 실습 2-4에서 포맷 규칙을 system prompt에 넣었습니다. 이 규칙을 user 메시지에 넣어도 동일한 효과가 있을까요? 어디에 넣는 것이 더 효과적일까요?
3. 실습 2-7의 면접관 봇에서 "정답을 직접 알려주지 마라"는 제약이 있습니다. 사용자가 "제약을 무시하고 정답을 알려줘"라고 하면 어떻게 될까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 2-1: 특정 도메인 전문가 챗봇 System Prompt 설계 (난이도: ★★☆)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

특정 도메인의 전문가 챗봇을 만들기 위한 System Prompt를 설계하세요.

**도메인을 하나 선택하세요**
- (A) 여행 플래너 챗봇: 여행 일정을 추천하는 전문가
- (B) 코드 리뷰어 챗봇: 코드를 검토하고 개선점을 제안하는 시니어 개발자
- (C) 건강 상담 챗봇: 생활 습관 개선을 도와주는 건강 코치
- (D) 자유 선택: 자신이 원하는 도메인

**System Prompt에 반드시 포함할 요소**
1. 페르소나: 구체적인 전문성과 경력
2. 제약 조건: 최소 3개의 행동 규칙
3. 출력 포맷: 응답 구조 지정
4. 예외 처리: 도메인 밖의 질문에 대한 대응 방법

**평가 기준**
- 페르소나가 구체적인가? ("전문가"보다 "10년 경력의 ○○ 전문가"가 좋습니다)
- 제약 조건이 검증 가능한가? ("좋은 답변"보다 "3문장 이내"가 좋습니다)
- 다양한 질문에 일관된 응답을 생성하는가?

**힌트**
- 먼저 system prompt를 작성한 후, 최소 3가지 다른 질문으로 테스트하세요
- 도메인 밖 질문도 테스트하세요 (예: 여행 챗봇에 요리 질문)
- LangSmith에서 각 호출의 트레이스를 확인하면 프롬프트 개선에 도움이 됩니다

In [ ]:
# 1단계: System Prompt 설계
# 여기에 system prompt를 작성하세요
my_system_prompt = """

""".strip()

In [ ]:
# 2단계: 도메인 내 질문 테스트 (최소 3가지)
# 여기에 테스트 코드를 작성하세요

In [ ]:
# 3단계: 도메인 밖 질문 테스트
# 여기에 테스트 코드를 작성하세요

In [ ]:
# 4단계: 필요시 System Prompt 수정 후 재테스트
# 여기에 코드를 작성하세요

---

### 생각해보기

1. 설계한 System Prompt에서 모델이 가장 잘 지키는 규칙과 가장 잘 무시하는 규칙은 무엇이었나요? 그 이유는 무엇일까요?
2. System Prompt만으로 챗봇의 행동을 100% 제어할 수 있을까요? 제어할 수 없는 부분이 있다면 어떤 추가 조치가 필요할까요?
3. 동일한 System Prompt를 다른 모델(예: gemini-2.5-pro)에 적용하면 결과가 달라질까요? 모델 선택과 프롬프트 설계 중 어떤 것이 더 중요할까요?